In [1]:
# output
from __future__ import annotations

import json
from pathlib import Path

from arcs import ArcTracker
from extract_llm import LLMExtractor
from lore_graph import LoreGraph
from rules import PlotHoleDetector
from semantic import SemanticContinuity
import pandas as pd

import html_report

STORY_DIR = Path("story")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)



/Users/aries_fitriawan/anaconda3/envs/pycon26/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
extractor = LLMExtractor(
    model_name="gpt-4o-mini",
    max_input_chars=12000,
)

In [3]:
import re

def auto_alias(name: str):
    # simple heuristic: take first capital chunk
    match = re.match(r"[A-Z][a-z]+", name)
    return match.group(0) if match else None
    
def build_auto_roster(extractions):
    """
    Build a clean, de-duplicated roster.
    Merge characters that reference each other via aliases.
    """

    alias_graph = {}

    # Step 1: collect all alias relationships
    for ext in extractions:
        for char in ext.characters:
            canonical = char.canonical.strip()
            aliases = [a.strip() for a in char.aliases if a.strip()]

            alias_graph.setdefault(canonical, set()).add(canonical)

            for alias in aliases:
                alias_graph.setdefault(canonical, set()).add(alias)
                alias_graph.setdefault(alias, set()).add(canonical)

    # Step 2: union-find / grouping
    visited = set()
    groups = []

    def dfs(name, group):
        if name in visited:
            return
        visited.add(name)
        group.add(name)

        for neighbor in alias_graph.get(name, []):
            dfs(neighbor, group)

    for name in alias_graph:
        if name not in visited:
            group = set()
            dfs(name, group)
            groups.append(group)

    # Step 3: choose canonical name per group
    roster = {}

    for group in groups:
        # choose best canonical:
        # rule: longest name OR most capitalized OR first seen
        canonical = sorted(group, key=lambda x: (-len(x), x))[0]

        aliases = sorted(n for n in group if n != canonical)

        roster[canonical] = aliases

    return roster

In [4]:
# Canonical character roster plus aliases.
# Untuk visual novel


def load_chapters(folder: Path) -> list[tuple[str, str, str]]:
    rows = []
    for path in sorted(folder.glob("*.md")):
        chapter_id = path.stem
        title = path.stem.replace("_", " ").title()
        text = path.read_text(encoding="utf-8")
        rows.append((chapter_id, title, text))
    return rows
    
def build_drift_comparisons(extractions):
    comparisons = []

    # Use first chapter as baseline
    early_texts = []

    for ext in extractions[:1]:
        early_texts.append(ext.synopsis or "")

    for ext in extractions:
        late_texts = [ext.synopsis or ""]

        comparisons.append(
            (
                ext.chapter_id,
                early_texts,
                late_texts,
            )
        )

    return comparisons

def main() -> None:
    import pandas as pd

    extractor = LLMExtractor(
        model_name="gpt-4o-mini",
        max_input_chars=12000,
    )

    semantic = SemanticContinuity(
        model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )

    lore = LoreGraph()
    detector = PlotHoleDetector()
    arcs = ArcTracker(roster=ROSTER, lang="id")

    chapter_texts: dict[str, str] = {}
    all_arc_rows = []

    for chapter_id, title, text in load_chapters(STORY_DIR):
        extraction = extractor.extract(chapter_id, title, text)

        lore.ingest(extraction)
        issues = detector.check(extraction, lore)

        chapter_texts[chapter_id] = text
        baseline = list(chapter_texts.values())[: min(3, len(chapter_texts))]
        similarity = semantic.centroid_similarity(baseline, [text])
        drift_score = 1.0 - similarity

        arc_df = arcs.analyze(chapter_id, text)

        if not arc_df.empty:
            all_arc_rows.extend(arc_df.to_dict(orient="records"))

        out_dir = OUTPUT_DIR / chapter_id
        out_dir.mkdir(exist_ok=True)

        (out_dir / "extraction.json").write_text(
            extraction.model_dump_json(indent=2),
            encoding="utf-8",
        )

        (out_dir / "issues.json").write_text(
            json.dumps([i.model_dump() for i in issues], indent=2, ensure_ascii=False),
            encoding="utf-8",
        )

        (out_dir / "semantic.json").write_text(
            json.dumps(
                {
                    "chapter_id": chapter_id,
                    "similarity_to_early_baseline": round(similarity, 4),
                    "drift_score": round(drift_score, 4),
                },
                indent=2,
                ensure_ascii=False,
            ),
            encoding="utf-8",
        )

        arc_df.to_csv(out_dir / "character_emotions.csv", index=False)

        if not arc_df.empty:
            for character in sorted(set(arc_df["character"].tolist())):
                arcs.plot_arc(
                    arc_df,
                    character=character,
                    out_path=str(out_dir / f"arc_{character}.png"),
                )

        arc_images = list(out_dir.glob("arc_*.png"))

        df_all_arcs = pd.DataFrame(all_arc_rows)

        if not df_all_arcs.empty and "chapter_id" in df_all_arcs.columns:
            df_arc = df_all_arcs[df_all_arcs["chapter_id"] == chapter_id].copy()
        else:
            df_arc = pd.DataFrame()

        html_report.make_chapter_html(
            chapter_id=chapter_id,
            title=title,
            synopsis=extraction.synopsis,
            drift_score=round(drift_score, 4),
            similarity_score=round(similarity, 4),
            issues=issues,
            arc_image_paths=[str(p.name) for p in arc_images],
            output_path=str(out_dir / f"{chapter_id}_analytics.html"),
            arc_df=df_arc,
            use_llm_arc_analysis=True,
            extraction=extraction,   # IMPORTANT
        )

        print(
            f"[{chapter_id}] drift={drift_score:.4f} "
            f"issues={len(issues)} "
            f"emotion_rows={len(arc_df)}"
        )

    if all_arc_rows:
        full_arc = pd.DataFrame(all_arc_rows)
        full_arc.to_csv(OUTPUT_DIR / "all_character_emotions.csv", index=False)

In [5]:
extractions = []

for chapter_id, title, text in load_chapters(STORY_DIR):
    extraction = extractor.extract(chapter_id, title, text)
    extractions.append(extraction)

# print(extractions)


In [6]:
ROSTER = build_auto_roster(extractions)
print(ROSTER)

{'WiraMadha': ['Wira'], 'Barda': [], 'Cinta Bumi': [], 'Ahol': [], 'Osmo': [], 'HoppyLite': [], 'DrakeCloud': ['Drake'], 'Sylvia': [], 'Kelabang Agni': [], 'Rere': [], 'Buto Kalong': []}


In [7]:
if __name__ == "__main__":
    main()

Loading weights: 100%|█████████████████████| 199/199 [00:00<00:00, 10682.23it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████████████████| 201/201 [00:00<00:00, 8634.67it/s]


Model emotion labels: {0: 'LABEL_0', 1: 'LABEL_1', 2: 'LABEL_2', 3: 'LABEL_3', 4: 'LABEL_4'}
[chapter_001] drift=0.0000 issues=0 emotion_rows=18
[chapter_002] drift=0.1749 issues=1 emotion_rows=56
[chapter_003] drift=0.1371 issues=9 emotion_rows=34
